In [ ]:
# from datasets import load_dataset

# # Загрузка датасета
# dataset = load_dataset("Vikhrmodels/russian_math")

# # Просмотр структуры
# print(dataset)
# print(dataset['train'][0])  # Пример первой записи

# # Сохранение локально (опционально)
# #dataset.save_to_disk("./russian_math_local")

Generating train split: 100%|██████████| 199/199 [00:00<00:00, 66070.33 examples/s]

DatasetDict({
    train: Dataset({
        features: ['task', 'solution', 'short answer', 'class', 'grade'],
        num_rows: 199
    })
})
{'task': 'Девять действительных a1, a2, ..., a9 образуют арифметическую прогрессию. Известно, что a9 в 3 раза больше среднего арифметического этих девяти чисел. Найдите a1, если известно, что a4 = 6.', 'solution': 'Пусть 𝑎 — первый член прогрессии, а 𝑑 — её разность, тогда девять членов прогрессии равны 𝑎, 𝑎 + 𝑑, 𝑎 + 2𝑑, …, 𝑎 + 8𝑑. \r\nСреднее арифметическое чисел в арифметической прогрессии, состоящей из нечётного числа членов, равно среднему из этих чисел, т. е. в данном случае 𝑎 + 4𝑑. \r\nПолучаем уравнение 𝑎 + 8𝑑 = 3(𝑎 + 4𝑑), откуда следует 𝑎 + 8𝑑 = 3𝑎 + 12𝑑 и 𝑎 = −2𝑑. Тогда 6 = 𝑎4 = 𝑎 + 3𝑑 = 𝑑. Значит, 𝑎1 = −2𝑑 = −12.', 'short answer': '-12', 'class': 'school', 'grade': 11}


In [ ]:
# import pandas as pd

In [ ]:
# df = pd.DataFrame(dataset['train'])

In [15]:
# df['task'][0]

# Отладка агентов

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

MODEL_ID = os.getenv("MODEL_ID", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "512"))
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

_llm = None

def get_llm():
    global _llm
    if _llm is None:
        import torch
        from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
        from huggingface_hub import login
        
        # Логинимся если есть токен
        if HUGGINGFACE_TOKEN:
            login(token=HUGGINGFACE_TOKEN)
        
        torch.cuda.empty_cache()
        
        pipeline = HuggingFacePipeline.from_model_id(
            model_id=MODEL_ID,
            task="text-generation",
            device=0,
            pipeline_kwargs=dict(
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                repetition_penalty=1.1,
            ),
        )
        _llm = ChatHuggingFace(llm=pipeline)
    
    return _llm

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage
#from config import get_llm

# Промпт на русском, без служебных токенов
SYSTEM_PROMPT = """Ты генератор математических задач. Придумай ОДНУ задачу по указанной теме.

Требования:
1. Задача должна быть реалистичной и понятной
2. Ответ — число или простая формула
3. Сложность: уровень ЕГЭ или школьной олимпиады

Формат ответа (строго соблюдай):
ЗАДАЧА: [текст задачи]
ОТВЕТ: [числовой ответ]"""

TOPIC_PROMPTS = {
    "алгебра": "Составь задачу по алгебре: уравнения, неравенства, функции или прогрессии.",
    "комбинаторика": "Составь задачу по комбинаторике: перестановки, сочетания или размещения.",
    "вероятность и статистика": "Составь задачу по теории вероятностей или статистике."
}

def extract_problem_answer(text: str) -> tuple:
    """Извлекает задачу и ответ из ответа модели"""
    # Убираем служебные токены если они есть
    text = text.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip()
    
    problem = ""
    answer = ""
    
    # Ищем по-русски
    if "ЗАДАЧА:" in text and "ОТВЕТ:" in text:
        parts = text.split("ОТВЕТ:")
        if len(parts) >= 2:
            problem_part = parts[0]
            answer = parts[1].strip().split("\n")[0]
            problem = problem_part.replace("ЗАДАЧА:", "").strip()
    # Fallback на английский если модель переключилась
    elif "PROBLEM:" in text and "ANSWER:" in text:
        parts = text.split("ANSWER:")
        if len(parts) >= 2:
            problem = parts[0].replace("PROBLEM:", "").strip()
            answer = parts[1].strip().split("\n")[0]
    
    # Если не распарсилось — берём всё как задачу, последнее число как ответ
    if not problem:
        lines = [l.strip() for l in text.split("\n") if l.strip() and not l.startswith("Ты ") and not l.startswith("Требования")]
        if len(lines) >= 2:
            problem = "\n".join(lines[:-1])
            answer = lines[-1]
        else:
            problem = text
            answer = "неизвестно"
    
    return problem, answer

class GeneratorAgent:
    def __init__(self):
        self.llm = get_llm()
    
    def generate(self, topic: str) -> dict:
        topic_hint = TOPIC_PROMPTS.get(topic, "Составь математическую задачу.")
        
        # Формируем сообщения вручную в формате TinyLlama
        # TinyLlama использует: <|system|>...<|user|>...<|assistant|>...
        
        system_msg = SystemMessage(content=SYSTEM_PROMPT)
        human_msg = HumanMessage(content=f"Тема: {topic}. {topic_hint}")
        
        response = self.llm.invoke([system_msg, human_msg])
        text = response.content
        
        # # Очистка от служебных токенов
        # text = text.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip()
        
        # problem, answer = extract_problem_answer(text)
        
        # return {
        #     "full_response": text,
        #     "problem": problem,
        #     "ground_truth": answer
        # }
        return response

In [3]:
agentGenerator = GeneratorAgent()

/home/tas/.cache/pypoetry/virtualenvs/ai-mas-hse-project-VtkJAIkS-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.18it/s]
Device set to use cuda:0


In [4]:
agent_answer = agentGenerator.generate("алгебра")

In [11]:
print(agent_answer.content)

<|startoftext|><|im_start|>system
Ты генератор математических задач. Придумай ОДНУ задачу по указанной теме.

Требования:
1. Задача должна быть реалистичной и понятной
2. Ответ — число или простая формула
3. Сложность: уровень ЕГЭ или школьной олимпиады

Формат ответа (строго соблюдай):
ЗАДАЧА: [текст задачи]
ОТВЕТ: [числовой ответ]<|im_end|>
<|im_start|>user
Тема: алгебра. Составь задачу по алгебре: уравнения, неравенства, функции или прогрессии.<|im_end|>
<|im_start|>assistant
ЗАДАЧА: Уравнение числовых линейных операций: найди значение \( x \) так что  
\[ 4(2x - 5) + 7 = 3(x + 1) - 2x \]
ОТВЕТ: 3

---

Как разумно подумать? Решить уравнение как только возможное с упрощения обхваток.  
[3]


In [11]:
print(agent_answer['full_response'])

<|startoftext|><|im_start|>system
Ты генератор математических задач. Придумай ОДНУ задачу по указанной теме.

Требования:
1. Задача должна быть реалистичной и понятной
2. Ответ — число или простая формула
3. Сложность: уровень ЕГЭ или школьной олимпиады

Формат ответа (строго соблюдай):
ЗАДАЧА: [текст задачи]
ОТВЕТ: [числовой ответ]<|im_end|>
<|im_start|>user
Тема: алгебра. Составь задачу по алгебре: уравнения, неравенства, функции или прогрессии.<|im_end|>
<|im_start|>assistant
ZÁDACH: Число *x* такое, что в решении системы уравнений имеет простое решение. Если уравнения будут так:  
\[
\begin{cases}
x + 2y = 7 \\
3x - y = 5
\end{cases}
\], подставляя выражение за *y* из второго уравнения во первого — как показать, количество решений в зависимости от определения *x*. Но для специального случая: найти значение *x*, при котором система имеет **решенное** (простое) решение. Поскольку это систему прямых процентов, её определить как точку пересечения.  

Можно попробовать другое подходе.  

In [ ]:
import pickle

with open("../data/final_dataset/list_dict_with_tasks.pkl", "rb") as file:
    list_json = pickle.load(file)

In [15]:
import pandas as pd

df = pd.DataFrame(list_json)

In [17]:
df.topic.value_counts()

topic
Алгебра и арифметика        71
Комбинаторика               46
Олимпиадные задачи          13
Вероятность и статистика    12
Математический анализ        5
Name: count, dtype: int64